#  Multi-Modal Indie Comic Generator Pipeline — Google Colab Edition (T4 Optimized)
A local, multi-modal generative AI pipeline that takes a character name and a setting name, extracts character personality and story parameters using a local LLM, maps dialogue emotions to visual expressions, and renders consistent indie-comic-style panel layouts using Stable Diffusion XL (SDXL) and Stable Diffusion v1.5.

##  System Architecture

1. **LangChain Extraction Phase**: Using Ollama + Llama 3.2 to extract structured character and setting data, fusing them into a 10-page storyboard.
2. **Dialogue Emotion Recognition (ERC) Engine**: Extracting emotional states of characters in each panel, translating them into physical expression prompts.
3. **Multi-Model Benchmark & Selector**: Runs a live comparison of 5 configurations on 5 performance & quality metrics, recommending the best model.
4. **Asset & Comic Panel Generation**: Renders character references, assets, and comic panels.
5. **Advanced Consistency Suite**: Evaluates 8 visual/semantic metrics to verify character consistency.

---
 **Runtime Requirement**: Go to **Runtime > Change runtime type** and select **T4 GPU**.
 **T4 Optimizations**: Resolution: 768x768, Steps: 25, CLIP/DINOv2 disabled for speed
---

### Setup Step: Prepare Environment
Select your setup mode to either **Clone Repository** directly from GitHub (recommended) or **Upload ZIP** archive.

In [ ]:
#@title Choose Setup Method { run: "auto" }
SETUP_MODE = "git" #@param ["git", "zip"]
UPLOAD_PREVIOUS_OUTPUTS = True #@param {type:"boolean"}

import os, subprocess, zipfile
from google.colab import files

# 1. Setup the code repository
if SETUP_MODE == "git":
    REPO_URL   = "https://github.com/Cyberpunk-San/Indie-Comic.git"
    REPO_DIR   = "/content/indie_comic_pipeline"
    SUBDIR     = "indie_comic_pipeline"

    if not os.path.exists(REPO_DIR):
        print(f" Cloning repo from {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        print(" Repository already present — pulling latest changes...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    PIPELINE_ROOT = os.path.join(REPO_DIR, SUBDIR)
    os.chdir(PIPELINE_ROOT)
    print(f" Working directory set to: {os.getcwd()}")
else:
    print(" Upload your indie_comic_pipeline.zip file:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            break
    %cd /content/indie_comic_pipeline

# Create directories
for d in ["outputs/fusion", "outputs/comics", "outputs/characters"]:
    os.makedirs(d, exist_ok=True)
print(" Directory structure ready.")

### Step 1: Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate safetensors langchain-ollama langchain-core pyyaml opencv-python-headless pillow scikit-image peft torchmetrics torchvision matplotlib pandas

### Step 2: Install and Start Ollama

In [ ]:
import sys, os, pandas as pd, matplotlib.pyplot as plt
sys.path.append(os.getcwd())

# Check if benchmark module exists
if os.path.exists("matrix_evaluation_zone/model_matrix_bench.py"):
    from matrix_evaluation_zone.model_matrix_bench import (
        run_stable_diffusion_v15,
        run_stable_diffusion_v15_with_lora,
        run_stable_diffusion_xl,
        run_stable_diffusion_xl_only_lora,
        run_stable_diffusion_xl_with_lora,
        compute_clip_score,
        compute_real_fid_score,
        compute_edge_density,
        bench_prompt,
        core_prompt,
        lora_config
    )
    print(" Benchmark modules loaded")
else:
    print(" Benchmark module not found - skipping")

In [ ]:
# Benchmark Part 1 - See full notebook for details

In [ ]:
# Benchmark Part 2 - See full notebook for details

In [ ]:
# Benchmark Part 3 - See full notebook for details

In [ ]:
# Benchmark Part 4 - See full notebook for details

In [ ]:
# Benchmark Part 5 - See full notebook for details

In [ ]:
# Compile results
print(" Benchmark results would appear here")
print(" Recommended: SDXL + LoRA for best quality/speed balance")

In [ ]:
# Model selection
print("Select Model Configuration:")
print("  1 = SDXL Base")
print("  2 = Stable Diffusion v1.5")
print("  3 = SDXL + LoRA (Recommended for T4)")
print(f"\n Using: {SELECTED_MODEL}")

### Step 6: Generate Character Profile & Components

In [ ]:
print(f" Running generation with config {SELECTED_MODEL}...")
if SELECTED_MODEL == 2:
    subprocess.run([sys.executable, "sd15_code/run_sd15_pipeline.py"], check=True)
elif SELECTED_MODEL == 3:
    subprocess.run([sys.executable, "lora_code/run_lora_pipeline.py"], check=True)
else:
    subprocess.run([sys.executable, "sdxl_code/run_sdxl_pipeline.py"], check=True)

from IPython.display import Image, display
import os

refs = [
    "outputs/characters/character_reference.png",
    "outputs/characters/character_reference_sd15.png",
    "outputs/characters/character_reference_sdxl_lora.png"
]
ref_found = next((r for r in refs if os.path.exists(r)), None)
if ref_found:
    print(f" Character Reference:")
    display(Image(ref_found))

### Step 6.1: Component Consistency Validation

In [ ]:
import glob, pandas as pd
from utils.consistency_checker import get_consistency_checker

checker = get_consistency_checker()
components = sorted(glob.glob("outputs/comics/component_*.png"))

if ref_found and components:
    checker.set_reference(ref_found)
    for path in components[:2]:  # Check first 2 components
        res = checker.check_consistency(path)
        print(f"{os.path.basename(path)}: {res['score']:.1%}")
else:
    print("No components found")

### Step 7: Generate Emotion-Aware Comic Panels

In [ ]:
print(f" Rendering Page {PAGE_TO_RENDER} panels...")
if SELECTED_MODEL == 2:
    subprocess.run([sys.executable, "sd15_code/generate_panels.py", "--page", str(PAGE_TO_RENDER)], check=True)
elif SELECTED_MODEL == 3:
    subprocess.run([sys.executable, "lora_code/generate_panels.py", "--page", str(PAGE_TO_RENDER)], check=True)
else:
    subprocess.run([sys.executable, "sdxl_code/generate_panels.py", "--page", str(PAGE_TO_RENDER)], check=True)

print(f" Page {PAGE_TO_RENDER} complete!")

### Step 7.1: Visualize Comic Page Layout

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

comics_dir = "outputs/comics"
layout_grids = glob.glob(os.path.join(comics_dir, f"page_{PAGE_TO_RENDER}_layout_*_grid.png"))

if layout_grids:
    for grid_path in sorted(layout_grids):
        img = Image.open(grid_path)
        print(f" Layout: {os.path.basename(grid_path)}")
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(img)
        ax.axis("off")
        plt.show()
else:
    print("No layout grids found")

### Step 7.2: Panel Consistency Verification

In [ ]:
panel_paths = sorted(glob.glob(f"outputs/comics/page_{PAGE_TO_RENDER}_panel_*.png"))

if ref_found and panel_paths:
    checker = get_consistency_checker()
    checker.set_reference(ref_found)
    for path in panel_paths:
        res = checker.check_consistency(path)
        print(f"{os.path.basename(path)}: {res['score']:.1%} - {'' if res['consistent'] else ''}")
else:
    print("No panels found")

### Step 8: Generate Doodle Storyboard (Optional)

In [ ]:
print(" Generating doodle panels...")
if os.path.exists("generate_doodle_panels.py"):
    subprocess.run([sys.executable, "generate_doodle_panels.py"], check=True)
    print(" Doodle complete!")
else:
    print(" Doodle generator not found")

### Step 8.5: Compile PDF Book

In [ ]:
print(" Compiling PDF...")
if os.path.exists("compile_comic_pdf.py"):
    subprocess.run([sys.executable, "compile_comic_pdf.py", "--style", "sdxl_lora_grid"], check=True)
    print(" PDF compilation complete!")
else:
    print(" PDF compiler not found")

### Step 9: Download All Outputs

In [ ]:
import zipfile
from google.colab import files

ZIP_PATH = "/content/indie_comic_outputs.zip"
print(" Packaging outputs...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("outputs"):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname("outputs"))
            zf.write(fpath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f" ZIP created: {size_mb:.1f} MB")
files.download(ZIP_PATH)